# OTP-스마트카드 유사도 검증
> 10개 OD pair 대상 6-Level 유사도 지표 + legGeometry 공간 유사도 검증

**검증 항목**:
1. 새 TCN 데이터 (정류장좌표시퀀스 포함) 확인
2. OTP legGeometry polyline 추출 검증
3. 레벨별 유사도 지표 통계 (mode / sequence / time / route / spatial)
4. 공간 유사도 변별력 (동일 vs 상이 노선/모드)
5. 가중치별 매칭 성공률/실패율
6. OD별 배정 결과

---
## 1. 설정

In [1]:
import pandas as pd
import numpy as np
import csv
import ijson
import os
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
%load_ext autoreload
%autoreload 2

from module.similarity import (
    parse_otp_itinerary,
    parse_smartcard_trip,
    deduplicate_itineraries,
    compute_all_metrics,
    compute_composite_similarity,
    compute_avg_hausdorff_similarity,
    _extract_sc_known_coords,
    grade_similarity,
)
from module.route_features import fix_missing_distances
from module.gtfs_lookup import GTFSRouteLookup

---
## 2. 데이터 로드

In [ ]:
# 설정
OTP_INPUT_CSV = '../../data/otp/input/otp_od_input_over13.csv'
OTP_JSON_PATH = '../../data/otp/output/similarity.json'
TCN_PATH = '../../data/tcn/20250217/TCN_20250217_route.parquet'
GTFS_DIR = '../../data/gtfs/a1'
MAX_ODS = 10

# GTFS 룩업 초기화 (캐시 있으면 수 초 이내)
gtfs = GTFSRouteLookup(GTFS_DIR)

# CSV에서 id -> od_pair 매핑
od_map = {}
with open(OTP_INPUT_CSV) as f:
    for i, row in enumerate(csv.DictReader(f)):
        od_map[i] = row['od_pair']
print(f'CSV OD pairs: {len(od_map):,}')

# OTP JSON 스트리밍 로드 (MAX_ODS개)
otp_results_raw = {}
loaded = 0
with open(OTP_JSON_PATH, 'rb') as f:
    for item in ijson.items(f, 'item', use_float=True):
        item_id = item.get('id')
        itins = item.get('data', {}).get('plan', {}).get('itineraries', [])
        if not itins:
            continue
        od_pair = od_map.get(item_id)
        if od_pair is None:
            continue
        for itin in itins:
            fix_missing_distances(itin)
        otp_results_raw[od_pair] = itins
        loaded += 1
        if loaded >= MAX_ODS:
            break

# 중복 제거 (>=2 대안)
otp_results = {}
for od_pair, itins in otp_results_raw.items():
    deduped = deduplicate_itineraries(itins)
    if len(deduped) >= 2:
        otp_results[od_pair] = deduped

print(f'OTP 로드: {loaded} ODs')
print(f'중복 제거 후 (>=2 대안): {len(otp_results)} ODs')
for od, itins in otp_results.items():
    print(f'  {od}: {len(itins)}개 대안')

In [4]:
# TCN 로드 (새 버전: 정류장좌표시퀀스 포함)
tcn = pd.read_parquet(TCN_PATH)
print(f'TCN 통행: {len(tcn):,}건')

# 새 컬럼 확인
has_coord_seq = '정류장lat시퀀스' in tcn.columns and '정류장lon시퀀스' in tcn.columns
print(f'정류장좌표시퀀스 존재: {has_coord_seq}')

if has_coord_seq:
    coord_lens = tcn['정류장lat시퀀스'].apply(len)
    print(f'좌표시퀀스 길이: 평균 {coord_lens.mean():.2f}, '
          f'2개(직통) {(coord_lens == 2).sum():,}, '
          f'3개+(환승) {(coord_lens >= 3).sum():,}')

# 테스트 대상 SC 추출
otp_od_set = set(otp_results.keys())
tcn_sample = tcn[tcn['od_pair'].isin(otp_od_set)]
print(f'\n테스트 대상 SC 통행: {len(tcn_sample):,}건')
print(tcn_sample['od_pair'].value_counts())

TCN 통행: 12,999,778건
정류장좌표시퀀스 존재: True
좌표시퀀스 길이: 평균 2.63, 2개(직통) 6,632,227, 3개+(환승) 6,367,551

테스트 대상 SC 통행: 62건
od_pair
10003_8001757    14
10003_8001761    13
10003_8001754    10
10003_8001755     9
10003_8001060     7
10003_8001758     5
10003_10700       2
10003_1457        1
10003_10661       1
Name: count, dtype: int64


---
## 3. legGeometry polyline 추출 검증

In [5]:
# OTP itinerary에서 route_polyline + GTFS 확장 + shape polyline 확인
for od_pair, itins in list(otp_results.items())[:3]:
    print(f'\nOD: {od_pair}')
    for i, itin in enumerate(itins):
        otp_parsed = parse_otp_itinerary(itin, gtfs_lookup=gtfs)
        polyline = otp_parsed.get('route_polyline', [])
        full_coords = otp_parsed.get('full_stop_coords', [])
        shape_poly = otp_parsed.get('shape_polyline', [])
        print(f'  경로 {i}: mode={otp_parsed["modes"]} routes={otp_parsed["routes"]}')
        print(f'    legGeometry polyline: {len(polyline)} 좌표', end='')
        if polyline:
            lats = [p[0] for p in polyline]
            lons = [p[1] for p in polyline]
            print(f'  lat [{min(lats):.4f}, {max(lats):.4f}] lon [{min(lons):.4f}, {max(lons):.4f}]')
        else:
            print()
        print(f'    GTFS 정류장 확장: {len(full_coords)} 정류장', end='')
        if full_coords:
            print(f'  [{full_coords[0][0]} → {full_coords[-1][0]}]')
        else:
            print(' (매칭 실패)')
        print(f'    GTFS shape polyline: {len(shape_poly)} 좌표', end='')
        if shape_poly:
            slats = [p[0] for p in shape_poly]
            slons = [p[1] for p in shape_poly]
            print(f'  lat [{min(slats):.4f}, {max(slats):.4f}] lon [{min(slons):.4f}, {max(slons):.4f}]')
        else:
            print(' (shape 없음)')


OD: 10003_10661
  경로 0: mode={'bus'} routes=['5531']
    legGeometry polyline: 316 좌표  lat [37.3529, 37.4636] lon [126.8976, 126.9492]
    GTFS 정류장 확장: 29 정류장  [군포1동행정복지센터.군포역 → 말미고개.금천소방서]
    GTFS shape polyline: 312 좌표  lat [37.3530, 37.4636] lon [126.8976, 126.9492]
  경로 1: mode={'train', 'bus'} routes=['5625', ' 1호선']
    legGeometry polyline: 118 좌표  lat [37.3529, 37.4636] lon [126.8976, 126.9485]
    GTFS 정류장 확장: 0 정류장 (매칭 실패)
    GTFS shape polyline: 0 좌표 (shape 없음)
  경로 2: mode={'train', 'bus'} routes=['5623', ' 1호선']
    legGeometry polyline: 118 좌표  lat [37.3529, 37.4636] lon [126.8976, 126.9485]
    GTFS 정류장 확장: 0 정류장 (매칭 실패)
    GTFS shape polyline: 0 좌표 (shape 없음)

OD: 10003_10700
  경로 0: mode={'bus'} routes=['5531']
    legGeometry polyline: 285 좌표  lat [37.3529, 37.4522] lon [126.9016, 126.9492]
    GTFS 정류장 확장: 27 정류장  [군포1동행정복지센터.군포역 → 시흥사거리]
    GTFS shape polyline: 281 좌표  lat [37.3530, 37.4522] lon [126.9017, 126.9492]
  경로 1: mode={'train', 'bus'} routes=[' 1호선',

---
## 4. SC 환승 좌표 추출 검증

In [6]:
# 환승 통행의 좌표 추출 확인 (GTFS 확장 + shape polyline 포함)
transfer_sample = tcn_sample[tcn_sample['환승횟수'] >= 1].head(10)
print(f'환승 통행 {len(transfer_sample)}건 샘플:\n')

for _, row in transfer_sample.iterrows():
    sc_parsed = parse_smartcard_trip(row, gtfs_lookup=gtfs)
    sc_coords = _extract_sc_known_coords(sc_parsed)
    full_coords = sc_parsed.get('full_stop_coords', [])
    shape_poly = sc_parsed.get('shape_polyline', [])
    print(f'OD: {row["od_pair"]}, 환승: {row["환승횟수"]}회')
    print(f'  정류장: {sc_parsed["stops"]}')
    print(f'  노선: {sc_parsed["routes"]}')
    print(f'  stop_coords(중간): {sc_parsed.get("stop_coords", [])}')
    print(f'  추출 좌표: {len(sc_coords)}개')
    print(f'  GTFS 정류장 확장: {len(full_coords)} 정류장', end='')
    if full_coords:
        print(f'  [{full_coords[0][0]} → {full_coords[-1][0]}]')
    else:
        print(' (매칭 실패)')
    print(f'  GTFS shape polyline: {len(shape_poly)} 좌표', end='')
    if shape_poly:
        slats = [p[0] for p in shape_poly]
        slons = [p[1] for p in shape_poly]
        print(f'  lat [{min(slats):.4f}, {max(slats):.4f}] lon [{min(slons):.4f}, {max(slons):.4f}]')
    else:
        print(' (shape 없음)')
    print()

환승 통행 2건 샘플:

OD: 10003_1457, 환승: 2회
  정류장: ['군포역', '금정역', '금정', '범계']
  노선: ['5531', '1호선']
  stop_coords(중간): [(37.37152, 126.94351), (37.371716, 126.943513)]
  추출 좌표: 4개
  GTFS 정류장 확장: 0 정류장 (매칭 실패)
  GTFS shape polyline: 0 좌표 (shape 없음)

OD: 10003_10700, 환승: 1회
  정류장: ['군포역', '서안양우체국', '시흥사거리']
  노선: ['5531', '5530']
  stop_coords(중간): [(37.39365, 126.92592)]
  추출 좌표: 3개
  GTFS 정류장 확장: 0 정류장 (매칭 실패)
  GTFS shape polyline: 0 좌표 (shape 없음)



---
## 5. 전체 유사도 계산 (Non-GTFS, legGeometry 기반)

In [7]:
# 가중치 설정
WEIGHTS = {
    'mode': 0.14, 'sequence': 0.21, 'time': 0.14,
    'route': 0.21, 'spatial': 0.30,
}
THRESHOLD = 0.6
print(f'가중치: {WEIGHTS}')
print(f'매칭 임계값: {THRESHOLD}')

가중치: {'mode': 0.14, 'sequence': 0.21, 'time': 0.14, 'route': 0.21, 'spatial': 0.3}
매칭 임계값: 0.6


In [8]:
# 전체 SC 통행 vs OTP 대안 매칭 (GTFS 확장 + shape polyline 포함)
all_results = []

for od_pair in tqdm(otp_results.keys(), desc='유사도 계산'):
    itins = otp_results[od_pair]
    sc_trips = tcn_sample[tcn_sample['od_pair'] == od_pair]
    if len(sc_trips) == 0:
        continue

    for _, sc_row in sc_trips.iterrows():
        sc_parsed = parse_smartcard_trip(sc_row, gtfs_lookup=gtfs)
        sc_coords = _extract_sc_known_coords(sc_parsed)

        scores = []
        for idx, itin in enumerate(itins):
            otp_parsed = parse_otp_itinerary(itin, gtfs_lookup=gtfs)
            metrics = compute_all_metrics(otp_parsed, sc_parsed)
            cs = compute_composite_similarity(metrics, weights=WEIGHTS)
            scores.append({
                'idx': idx,
                'metrics': metrics,
                **cs,
                'route_match': 1 if set(otp_parsed['routes']) & set(sc_parsed['routes']) else 0,
                'mode_match': 1 if otp_parsed['modes'] == sc_parsed['modes'] else 0,
                'has_gtfs': 1 if otp_parsed.get('full_stop_coords') and sc_parsed.get('full_stop_coords') else 0,
                'has_shape': 1 if otp_parsed.get('shape_polyline') and sc_parsed.get('shape_polyline') else 0,
                'spatial_method': metrics.get('spatial_method', 'none'),
                'otp_shape_pts': len(otp_parsed.get('shape_polyline', [])),
                'sc_shape_pts': len(sc_parsed.get('shape_polyline', [])),
            })

        best = max(scores, key=lambda x: x['composite'])

        all_results.append({
            'od_pair': od_pair,
            'sc_category': sc_row.get('transport_category', ''),
            'sc_transfers': sc_parsed['transfer_count'],
            'sc_coords_count': len(sc_coords),
            'matched': best['composite'] >= THRESHOLD,
            'best_composite': round(best['composite'], 4),
            'best_idx': best['idx'],
            'mode_score': round(best['mode_score'], 4),
            'sequence_score': round(best['sequence_score'], 4),
            'time_score': round(best['time_score'], 4),
            'route_score': round(best['route_score'], 4),
            'spatial_score': round(best['spatial_score'], 4),
            'best_route_match': best['route_match'],
            'best_mode_match': best['mode_match'],
            'has_gtfs': best['has_gtfs'],
            'has_shape': best['has_shape'],
            'spatial_method': best['spatial_method'],
            'otp_shape_pts': best['otp_shape_pts'],
            'sc_shape_pts': best['sc_shape_pts'],
        })

results_df = pd.DataFrame(all_results)
print(f'총 SC 통행: {len(results_df):,}건')

gtfs_count = results_df['has_gtfs'].sum()
shape_count = results_df['has_shape'].sum()
print(f'GTFS 정류장 확장 성공: {gtfs_count:,}건 ({gtfs_count/len(results_df)*100:.1f}%)')
print(f'Shape polyline 확장 성공: {shape_count:,}건 ({shape_count/len(results_df)*100:.1f}%)')

print(f'\n=== 공간 유사도 측정 방법 분포 ===')
print(results_df['spatial_method'].value_counts().to_string())

print(f'\n=== Shape polyline 좌표 수 통계 (shape 있는 건만) ===')
shape_rows = results_df[results_df['has_shape'] == 1]
if len(shape_rows) > 0:
    print(f'  OTP shape: mean={shape_rows["otp_shape_pts"].mean():.0f}, '
          f'min={shape_rows["otp_shape_pts"].min()}, max={shape_rows["otp_shape_pts"].max()}')
    print(f'  SC shape:  mean={shape_rows["sc_shape_pts"].mean():.0f}, '
          f'min={shape_rows["sc_shape_pts"].min()}, max={shape_rows["sc_shape_pts"].max()}')

유사도 계산: 100%|██████████| 10/10 [00:00<00:00, 29.80it/s]

총 SC 통행: 62건
GTFS 정류장 확장 성공: 0건 (0.0%)
Shape polyline 확장 성공: 0건 (0.0%)

=== 공간 유사도 측정 방법 분포 ===
spatial_method
points_on_path    62

=== Shape polyline 좌표 수 통계 (shape 있는 건만) ===


---
## 6. 매칭 성공률 / 실패율

In [9]:
total = len(results_df)
matched = results_df['matched'].sum()
failed = total - matched

print(f'=== 매칭 결과 (threshold={THRESHOLD}) ===')
print(f'총 SC 통행: {total:,}')
print(f'매칭 성공:  {matched:,} ({matched/total*100:.1f}%)')
print(f'매칭 실패:  {failed:,} ({failed/total*100:.1f}%)')

print(f'\n=== 등급 분포 ===')
results_df['grade'] = results_df['best_composite'].apply(grade_similarity)
print(results_df['grade'].value_counts().to_string())

print(f'\n=== 임계값별 매칭률 ===')
for th in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    n = (results_df['best_composite'] >= th).sum()
    print(f'  >= {th}: {n:,}/{total:,} ({n/total*100:.1f}%)')

=== 매칭 결과 (threshold=0.6) ===
총 SC 통행: 62
매칭 성공:  62 (100.0%)
매칭 실패:  0 (0.0%)

=== 등급 분포 ===
grade
우수    61
양호     1

=== 임계값별 매칭률 ===
  >= 0.4: 62/62 (100.0%)
  >= 0.5: 62/62 (100.0%)
  >= 0.6: 62/62 (100.0%)
  >= 0.7: 62/62 (100.0%)
  >= 0.8: 61/62 (98.4%)
  >= 0.9: 60/62 (96.8%)


---
## 7. 레벨별 유사도 통계

In [10]:
level_cols = ['mode_score', 'sequence_score', 'time_score', 'route_score', 'spatial_score']

print('=== 레벨별 유사도 통계 (전체) ===')
print(results_df[level_cols + ['best_composite']].describe().round(4).to_string())

print(f'\n=== 레벨별 평균 ===')
for col in level_cols:
    mean = results_df[col].mean()
    std = results_df[col].std()
    print(f'  {col:20s}: {mean:.4f} (std={std:.4f})')
print(f'  {"best_composite":20s}: {results_df["best_composite"].mean():.4f} '
      f'(std={results_df["best_composite"].std():.4f})')

# 매칭 성공 vs 실패 비교
print(f'\n=== 매칭 성공 vs 실패 레벨별 평균 ===')
m = results_df[results_df['matched']]
f = results_df[~results_df['matched']]
print(f'{"":20s} {"성공(n="+str(len(m))+")":>15s} {"실패(n="+str(len(f))+")":>15s} {"차이":>10s}')
for col in level_cols + ['best_composite']:
    m_val = m[col].mean() if len(m) > 0 else 0
    f_val = f[col].mean() if len(f) > 0 else 0
    diff = m_val - f_val
    name = col.replace('_score', '').replace('best_', '')
    print(f'  {name:20s} {m_val:>14.4f} {f_val:>14.4f} {diff:>+10.4f}')

=== 레벨별 유사도 통계 (전체) ===
       mode_score  sequence_score  time_score  route_score  spatial_score  best_composite
count        62.0         62.0000     62.0000      62.0000        62.0000         62.0000
mean          1.0          0.9906      0.8092       0.9812         0.9999          0.9673
std           0.0          0.0525      0.1376       0.1050         0.0005          0.0361
min           1.0          0.6667      0.3520       0.3333         0.9965          0.7802
25%           1.0          1.0000      0.6932       1.0000         1.0000          0.9563
50%           1.0          1.0000      0.8232       1.0000         1.0000          0.9752
75%           1.0          1.0000      0.9247       1.0000         1.0000          0.9891
max           1.0          1.0000      1.0000       1.0000         1.0000          1.0000

=== 레벨별 평균 ===
  mode_score          : 1.0000 (std=0.0000)
  sequence_score      : 0.9906 (std=0.0525)
  time_score          : 0.8092 (std=0.1376)
  route_score     

---
## 8. 공간 유사도 변별력 분석

In [11]:
# SC 좌표 수 분포
print('=== SC 좌표 수 분포 ===')
print(results_df['sc_coords_count'].value_counts().sort_index().to_string())

# 전체 공간 유사도 통계
print(f'\n=== 공간 유사도 (spatial_score) 통계 ===')
print(results_df['spatial_score'].describe().round(4).to_string())

# Shape 유무별 공간 유사도 비교
if 'has_shape' in results_df.columns:
    shape_yes = results_df[results_df['has_shape'] == 1]['spatial_score']
    shape_no = results_df[results_df['has_shape'] == 0]['spatial_score']
    print(f'\n=== Shape polyline 유무별 공간 유사도 ===')
    print(f'  Shape 있음 (dense Hausdorff): mean={shape_yes.mean():.4f} (n={len(shape_yes)})')
    if len(shape_no) > 0:
        print(f'  Shape 없음 (fallback):        mean={shape_no.mean():.4f} (n={len(shape_no)})')

# GTFS 유무별 공간 유사도 비교
if 'has_gtfs' in results_df.columns:
    gtfs_yes = results_df[results_df['has_gtfs'] == 1]['spatial_score']
    gtfs_no = results_df[results_df['has_gtfs'] == 0]['spatial_score']
    print(f'\n=== GTFS 정류장 확장 유무별 공간 유사도 ===')
    print(f'  GTFS 있음 (Hausdorff): mean={gtfs_yes.mean():.4f} (n={len(gtfs_yes)})')
    if len(gtfs_no) > 0:
        print(f'  GTFS 없음 (points-on-path): mean={gtfs_no.mean():.4f} (n={len(gtfs_no)})')

# 측정 방법별 공간 유사도
if 'spatial_method' in results_df.columns:
    print(f'\n=== 측정 방법별 공간 유사도 ===')
    for method, group in results_df.groupby('spatial_method'):
        print(f'  {method:20s}: mean={group["spatial_score"].mean():.4f} '
              f'std={group["spatial_score"].std():.4f} (n={len(group)})')

# 변별력: 노선 일치 vs 불일치
print(f'\n--- 전체 대안 비교 변별력 ---')
pair_scores = []
for od_pair in otp_results.keys():
    itins = otp_results[od_pair]
    sc_trips = tcn_sample[tcn_sample['od_pair'] == od_pair]
    for _, sc_row in sc_trips.head(10).iterrows():
        sc_parsed = parse_smartcard_trip(sc_row, gtfs_lookup=gtfs)
        for idx, itin in enumerate(itins):
            otp_parsed = parse_otp_itinerary(itin, gtfs_lookup=gtfs)
            metrics = compute_all_metrics(otp_parsed, sc_parsed)
            pair_scores.append({
                'spatial': metrics.get('polyline_similarity', 0),
                'spatial_method': metrics.get('spatial_method', 'none'),
                'route_match': 1 if set(otp_parsed['routes']) & set(sc_parsed['routes']) else 0,
                'mode_match': 1 if otp_parsed['modes'] == sc_parsed['modes'] else 0,
                'sc_coords': len(_extract_sc_known_coords(sc_parsed)),
                'has_gtfs': 1 if otp_parsed.get('full_stop_coords') and sc_parsed.get('full_stop_coords') else 0,
                'has_shape': 1 if otp_parsed.get('shape_polyline') and sc_parsed.get('shape_polyline') else 0,
            })

pair_df = pd.DataFrame(pair_scores)
print(f'총 비교 쌍: {len(pair_df):,}')

same_r = pair_df[pair_df['route_match'] == 1]['spatial']
diff_r = pair_df[pair_df['route_match'] == 0]['spatial']
print(f'\n[노선 기반 변별력]')
print(f'  동일 노선: mean={same_r.mean():.4f} (n={len(same_r)})')
print(f'  상이 노선: mean={diff_r.mean():.4f} (n={len(diff_r)})')
if len(same_r) > 0 and len(diff_r) > 0:
    print(f'  차이: {same_r.mean() - diff_r.mean():.4f}')

same_m = pair_df[pair_df['mode_match'] == 1]['spatial']
diff_m = pair_df[pair_df['mode_match'] == 0]['spatial']
print(f'\n[모드 기반 변별력]')
print(f'  동일 모드: mean={same_m.mean():.4f} (n={len(same_m)})')
print(f'  상이 모드: mean={diff_m.mean():.4f} (n={len(diff_m)})')
if len(same_m) > 0 and len(diff_m) > 0:
    print(f'  차이: {same_m.mean() - diff_m.mean():.4f}')

# Shape 기반 vs 비-Shape 기반 변별력 비교
shape_pairs = pair_df[pair_df['has_shape'] == 1]
non_shape_pairs = pair_df[pair_df['has_shape'] == 0]
if len(shape_pairs) > 0:
    s_same = shape_pairs[shape_pairs['route_match'] == 1]['spatial']
    s_diff = shape_pairs[shape_pairs['route_match'] == 0]['spatial']
    print(f'\n[Shape 기반 노선 변별력]')
    print(f'  동일 노선: mean={s_same.mean():.4f} (n={len(s_same)})')
    if len(s_diff) > 0:
        print(f'  상이 노선: mean={s_diff.mean():.4f} (n={len(s_diff)})')
        print(f'  차이: {s_same.mean() - s_diff.mean():.4f}')

# 환승 통행만
transfer_pairs = pair_df[pair_df['sc_coords'] >= 3]
if len(transfer_pairs) > 0:
    same_rt = transfer_pairs[transfer_pairs['route_match'] == 1]['spatial']
    diff_rt = transfer_pairs[transfer_pairs['route_match'] == 0]['spatial']
    print(f'\n[환승 통행만 (좌표 3개+) 노선 변별력]')
    print(f'  동일 노선: mean={same_rt.mean():.4f} (n={len(same_rt)})')
    print(f'  상이 노선: mean={diff_rt.mean():.4f} (n={len(diff_rt)})')
    if len(same_rt) > 0 and len(diff_rt) > 0:
        print(f'  차이: {same_rt.mean() - diff_rt.mean():.4f}')

=== SC 좌표 수 분포 ===
sc_coords_count
2    60
3     1
4     1

=== 공간 유사도 (spatial_score) 통계 ===
count    62.0000
mean      0.9999
std       0.0005
min       0.9965
25%       1.0000
50%       1.0000
75%       1.0000
max       1.0000

=== Shape polyline 유무별 공간 유사도 ===
  Shape 있음 (dense Hausdorff): mean=nan (n=0)
  Shape 없음 (fallback):        mean=0.9999 (n=62)

=== GTFS 정류장 확장 유무별 공간 유사도 ===
  GTFS 있음 (Hausdorff): mean=nan (n=0)
  GTFS 없음 (points-on-path): mean=0.9999 (n=62)

=== 측정 방법별 공간 유사도 ===
  points_on_path      : mean=0.9999 std=0.0005 (n=62)

--- 전체 대안 비교 변별력 ---
총 비교 쌍: 155

[노선 기반 변별력]
  동일 노선: mean=0.9986 (n=59)
  상이 노선: mean=0.9974 (n=96)
  차이: 0.0013

[모드 기반 변별력]
  동일 모드: mean=0.9999 (n=127)
  상이 모드: mean=0.9888 (n=28)
  차이: 0.0111

[환승 통행만 (좌표 3개+) 노선 변별력]
  동일 노선: mean=0.9801 (n=4)
  상이 노선: mean=0.9375 (n=4)
  차이: 0.0425


---
## 9. OD별 매칭 결과

In [12]:
# OD별 요약
od_summary = results_df.groupby('od_pair').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
    avg_composite=('best_composite', 'mean'),
    avg_mode=('mode_score', 'mean'),
    avg_sequence=('sequence_score', 'mean'),
    avg_time=('time_score', 'mean'),
    avg_route=('route_score', 'mean'),
    avg_spatial=('spatial_score', 'mean'),
).round(4)
od_summary['match_rate'] = (od_summary['matched'] / od_summary['total']).round(4)
od_summary['failed'] = od_summary['total'] - od_summary['matched']

print('=== OD별 매칭 결과 ===')
display_cols = ['total', 'matched', 'failed', 'match_rate',
                'avg_composite', 'avg_mode', 'avg_sequence',
                'avg_time', 'avg_route', 'avg_spatial']
print(od_summary[display_cols].to_string())

=== OD별 매칭 결과 ===
               total  matched  failed  match_rate  avg_composite  avg_mode  avg_sequence  avg_time  avg_route  avg_spatial
od_pair                                                                                                                   
10003_10661        1        1       0         1.0         0.9893       1.0        1.0000    0.9237     1.0000       1.0000
10003_10700        2        2       0         1.0         0.9092       1.0        0.8334    0.9788     0.7500       0.9988
10003_1457         1        1       0         1.0         0.7802       1.0        0.7500    0.8124     0.3333       0.9965
10003_8001060      7        7       0         1.0         0.9403       1.0        1.0000    0.5737     1.0000       1.0000
10003_8001754     10       10       0         1.0         0.9832       1.0        1.0000    0.8800     1.0000       1.0000
10003_8001755      9        9       0         1.0         0.9720       1.0        1.0000    0.8001     1.0000       1.000

---
## 10. OD별 경로 배정 결과

In [13]:
# OD별 OTP 경로 배정 상세
for od_pair in otp_results.keys():
    itins = otp_results[od_pair]
    od_sub = results_df[results_df['od_pair'] == od_pair]
    if len(od_sub) == 0:
        continue

    total = len(od_sub)
    matched_sub = od_sub[od_sub['matched']]
    unmatched_sub = od_sub[~od_sub['matched']]

    print(f'--- {od_pair} (SC {total}건) ---')

    # OTP 경로 요약 + 배정
    for i, itin in enumerate(itins):
        otp_p = parse_otp_itinerary(itin)
        n = len(matched_sub[matched_sub['best_idx'] == i])
        prob = n / total
        print(f'  [{i}] {sorted(otp_p["modes"])} {otp_p["routes"]} '
              f'환승={otp_p["transfer_count"]}회 '
              f'→ {n}건 ({prob*100:.1f}%)')

    n_other = len(unmatched_sub)
    print(f'  [other] → {n_other}건 ({n_other/total*100:.1f}%)')
    print()

--- 10003_10661 (SC 1건) ---
  [0] ['bus'] ['5531'] 환승=0회 → 1건 (100.0%)
  [1] ['bus', 'train'] ['5625', ' 1호선'] 환승=1회 → 0건 (0.0%)
  [2] ['bus', 'train'] ['5623', ' 1호선'] 환승=1회 → 0건 (0.0%)
  [other] → 0건 (0.0%)

--- 10003_10700 (SC 2건) ---
  [0] ['bus'] ['5531'] 환승=0회 → 2건 (100.0%)
  [1] ['bus', 'train'] [' 1호선', '5531'] 환승=1회 → 0건 (0.0%)
  [2] ['bus', 'train'] [' 1호선', '5531'] 환승=1회 → 0건 (0.0%)
  [other] → 0건 (0.0%)

--- 10003_1457 (SC 1건) ---
  [0] ['bus'] ['542'] 환승=0회 → 0건 (0.0%)
  [1] ['bus', 'train'] [' 4호선', '542'] 환승=1회 → 0건 (0.0%)
  [2] ['bus', 'train'] [' 4호선', '5531'] 환승=1회 → 1건 (100.0%)
  [3] ['bus'] ['60'] 환승=0회 → 0건 (0.0%)
  [4] ['bus', 'train'] [' 4호선', '8000'] 환승=1회 → 0건 (0.0%)
  [other] → 0건 (0.0%)

--- 10003_8001060 (SC 7건) ---
  [0] ['bus'] ['8-2'] 환승=0회 → 0건 (0.0%)
  [1] ['bus'] ['5531'] 환승=0회 → 7건 (100.0%)
  [2] ['bus'] ['8'] 환승=0회 → 0건 (0.0%)
  [3] ['bus'] ['5'] 환승=0회 → 0건 (0.0%)
  [other] → 0건 (0.0%)

--- 10003_8001754 (SC 10건) ---
  [0] ['bus'] ['5531'] 환승=0회 → 10

---
## 11. 교통수단 카테고리별 매칭률

In [14]:
cat_summary = results_df.groupby('sc_category').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
    avg_composite=('best_composite', 'mean'),
    avg_spatial=('spatial_score', 'mean'),
).round(4)
cat_summary['match_rate'] = (cat_summary['matched'] / cat_summary['total']).round(4)

print('=== 교통수단 카테고리별 매칭률 ===')
print(cat_summary.to_string())

# 환승 여부별
results_df['has_transfer'] = results_df['sc_transfers'] > 0
tr_summary = results_df.groupby('has_transfer').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
    avg_composite=('best_composite', 'mean'),
    avg_spatial=('spatial_score', 'mean'),
).round(4)
tr_summary['match_rate'] = (tr_summary['matched'] / tr_summary['total']).round(4)

print(f'\n=== 환승 여부별 매칭률 ===')
print(tr_summary.to_string())

=== 교통수단 카테고리별 매칭률 ===
             total  matched  avg_composite  avg_spatial  match_rate
sc_category                                                        
bus+train        1        1         0.7802       0.9965         1.0
bus_only        61       61         0.9704       1.0000         1.0

=== 환승 여부별 매칭률 ===
              total  matched  avg_composite  avg_spatial  match_rate
has_transfer                                                        
False            60       60         0.9728        1.000         1.0
True              2        2         0.8017        0.997         1.0
